# Redrob Candidate Ranker Sandbox

This Google Colab notebook is the hosted sandbox/demo for the Redrob Intelligent Candidate Discovery challenge.

It runs the same deterministic scoring logic on the organizer-provided `sample_candidates.json` file and writes a ranked CSV with the required columns:

`candidate_id,rank,score,reasoning`

The full hackathon submission is generated locally from all 100,000 candidates using the command in `README.md`.

In [ ]:
# Clone the public repository. If you already have it mounted in Colab,
# you can skip this cell and cd into the repo folder manually.
import os

if not os.path.exists('/content/Redrob-AI-candidate_Ranker'):
    !git clone https://github.com/Tanvii13/Redrob-AI-candidate_Ranker.git

%cd /content/Redrob-AI-candidate_Ranker

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import csv
import json
from pathlib import Path

from main import normalize_scores, reasoning, score_candidate

sample_path = Path('data/sample_candidates.json')
out_path = Path('sandbox_sample_ranking.csv')

with sample_path.open('r', encoding='utf-8') as handle:
    candidates = json.load(handle)

scored = []
kept = {}
for candidate in candidates:
    row = score_candidate(candidate)
    scored.append(row)
    kept[row['candidate_id']] = candidate

scored.sort(key=lambda row: (-row['score'], row['candidate_id']))
ranked = normalize_scores(scored)

with out_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['candidate_id', 'rank', 'score', 'reasoning'])
    writer.writeheader()
    for rank, row in enumerate(ranked, start=1):
        candidate = kept[row['candidate_id']]
        writer.writerow({
            'candidate_id': row['candidate_id'],
            'rank': rank,
            'score': row['normalized_score'],
            'reasoning': reasoning(candidate, row),
        })

print(f'Ranked {len(ranked)} sample candidates.')
print(f'Wrote {out_path.resolve()}')

In [ ]:
import pandas as pd

df = pd.read_csv('sandbox_sample_ranking.csv')
df.head(20)

## Notes for reviewers

- This sandbox ranks the small organizer sample so it runs quickly in a hosted CPU environment.
- The full reproduction command for all 100,000 candidates is in `README.md`.
- The scoring logic is deterministic and does not call external APIs or hosted LLMs during ranking.